In [2]:
import numpy as np
from scipy import signal, integrate
import warnings
from scipy.integrate import IntegrationWarning

# Ignore singularity warnings from integration
warnings.filterwarnings("ignore", category=IntegrationWarning)

# =========================================================
# 1. DYNAMIC INPUT PARAMETER SETTINGS
# =========================================================
E_max = 150.0             # Maximum projectile energy (in MeV)
sobp_percentage = 25.0    # SOBP plateau width as a percentage of total range R0 (%)

# Medium Physics Parameters (PMMA / Plexiglass)
alpha = 0.00185           # Empirical Bortfeld constant for PMMA
p = 1.77                  # Empirical Bortfeld exponent for PMMA
D0 = 1.0                  # Target SOBP dose (Relative)

# Maximum particle scale for TOPAS simulation
max_particles = 100000

# =========================================================
# 2. AUTOMATIC CALCULATION: RANGE (R0) & SOBP WIDTH (cm)
# =========================================================
# Calculate R0 (Distal Edge / Maximum depth of Pristine Peak)
R0 = alpha * (E_max ** p)
db = R0

# Calculate SOBP width in cm based on the percentage
sobp_width = R0 * (sobp_percentage / 100.0)

# Calculate Proximal Edge (Initial depth of the tumor target)
da = db - sobp_width

print(f"===========================================================")
print(f" TUMOR TARGET SUMMARY")
print(f"===========================================================")
print(f"Max Energy (E_max)    : {E_max} MeV")
print(f"Max Range (R0/db)     : {R0:.3f} cm")
print(f"SOBP Width ({sobp_percentage}%)      : {sobp_width:.3f} cm")
print(f"Tumor Target Range    : {da:.3f} cm to {db:.3f} cm\n")


# =========================================================
# 3. BORTFELD ANALYTICAL RANGE-ENERGY CALCULATION
# =========================================================
a = alpha ** (1.0 / p)
q = 1.0 - 1.0 / p

def g(d):
    return np.piecewise(d, [d < 0, d > 0], [0, lambda d: 1.0 / (p * a * d ** q)])

def bragg_peak(R, d):
    return g(R - d)

def average(func, d0, d1):
    integral, _ = integrate.quad(func, d0, d1) / (d1 - d0)
    return integral

def impulse(func, x):
    h = np.zeros(len(x))
    dx = x[1] - x[0]
    h[0] = average(func, 0, 0.5 * dx)
    for n in range(1, len(x)):
        h[n] = average(func, x[n] - 0.5 * dx, x[n] + 0.5 * dx)
    return h

def W(R):
    return np.piecewise(R, [(da <= R) & (R < db)], [lambda R: D0 * p * np.sin(np.pi / p) * a / (np.pi * (db - R) ** (1.0 / p)), 0])

def back_transform(w, da, db, x):
    N = len(x)
    dx = x[1] - x[0]
    Na = int(da / dx)
    Nb = int(db / dx)
    w = w[:Nb + 1]
    w_reverse = w[::-1]
    return np.concatenate((np.zeros(Na), w_reverse[Na:Nb + 1], np.zeros(N - Nb - 1)))

# Grid Definition
N_grid = 201
dmax = 20.0
d = np.linspace(0, dmax, N_grid)
dd = d[1] - d[0]
Na = int(da / dd)
Nb = int(db / dd)

g_avs = impulse(g, d)
M_grid = N_grid
d_w = np.arange(M_grid) * dd
Nd = M_grid + N_grid - 1

yd = D0 * np.ones(Nd)

# Signal Deconvolution
w, remainder = signal.deconvolve(yd, g_avs)
w /= dd
w2 = back_transform(w, da, db, d)


# =========================================================
# 4. VARIAN PROBEAM OPTICAL CHARACTERISTICS FUNCTION
# =========================================================
def calculate_probeam_specs(energy_mev):
    """
    Calculates the spatial and angular spread for Varian ProBeam
    based on relativistic momentum and predefined machine constants.
    """
    a0 = 2.60137049
    a1 = 1467.402324
    b0 = 1.13883335
    proton_mass = 938.272 # Rest mass of proton (MeV/c^2)

    # Relativistic momentum and velocity (beta)
    momentum = np.sqrt(energy_mev**2 + 2 * energy_mev * proton_mass)
    beta = momentum / (energy_mev + proton_mass)

    # 1. Beam Position Spread (sigma_0 in mm)
    pos_spread = np.sqrt(a0**2 + (a1 / momentum)**2)

    # 2. Beam Angular Spread (sigma_p in mrad)
    ang_spread = (b0 / (beta * momentum)) * 1000

    return pos_spread, ang_spread


# =========================================================
# 5. DATA EXTRACTION & COMPILATION FOR TOPAS BASH SCRIPT
# =========================================================
beam_ranges = d[Na:Nb+1].copy()
beam_weights_raw = w2[Na:Nb+1]

# Force the last range value to exactly match db
# to ensure precise reverse calculation of E_max
beam_ranges[-1] = db

# Convert range (cm) back to Energy (MeV)
beam_energies = (beam_ranges / alpha) ** (1.0 / p)

# Normalize particle weights to TOPAS maximum limit
max_weight = np.max(beam_weights_raw)
particle_weights = np.round((beam_weights_raw / max_weight) * max_particles).astype(int)

# Storage arrays for output format
list_energies = []
list_weights = []
list_pos = []
list_ang = []

# Loop backwards (from deepest penetration energy to shallowest)
for i in range(len(beam_ranges) - 1, -1, -1):
    e_val = beam_energies[i]
    w_val = particle_weights[i]
    pos_val, ang_val = calculate_probeam_specs(e_val)

    list_energies.append(e_val)
    list_weights.append(w_val)
    list_pos.append(pos_val)
    list_ang.append(ang_val)


# =========================================================
# 6. PRINT RESULTS (READY TO COPY-PASTE TO BASH SCRIPT)
# =========================================================
print(f"===========================================================")
print(f" EXTRACTION RESULTS : {len(list_energies)} ENERGY LAYERS")
print(f"===========================================================\n")

print("# 1. ENERGIES (MeV)")
for val in list_energies: print(f"{val:.6f}")

print("\n# 2. WEIGHTS / PARTICLES")
for val in list_weights: print(f"{val}")

print("\n# 3. POSITION SPREADS (mm)")
for val in list_pos: print(f"{val:.3f}")

print("\n# 4. ANGULAR SPREADS (mrad)")
for val in list_ang: print(f"{val:.2f}")

 TUMOR TARGET SUMMARY
Max Energy (E_max)    : 150.0 MeV
Max Range (R0/db)     : 13.148 cm
SOBP Width (25.0%)      : 3.287 cm
Tumor Target Range    : 9.861 cm to 13.148 cm

 EXTRACTION RESULTS : 34 ENERGY LAYERS

# 1. ENERGIES (MeV)
150.000000
149.044962
148.396135
147.745115
147.091879
146.436402
145.778657
145.118620
144.456263
143.791559
143.124480
142.454999
141.783086
141.108713
140.431848
139.752463
139.070524
138.386001
137.698861
137.009070
136.316595
135.621400
134.923451
134.222710
133.519141
132.812706
132.103365
131.391080
130.675808
129.957510
129.236141
128.511658
127.784016
127.053170

# 2. WEIGHTS / PARTICLES
100000
56990
44371
37194
32483
29101
26530
24493
22832
21446
20266
19248
18359
17573
16873
16244
15675
15158
14684
14250
13848
13476
13131
12808
12507
12224
11958
11707
11470
11246
11034
10832
10640
10457

# 3. POSITION SPREADS (mm)
3.722
3.728
3.733
3.737
3.742
3.746
3.751
3.756
3.761
3.766
3.770
3.775
3.780
3.786
3.791
3.796
3.801
3.806
3.812
3.817
3.823
3.829
3.8